In [4]:
from pathlib import Path
import numpy as np
%load_ext autoreload
%autoreload 2


In [5]:
# Cell 0 — CONFIG
import sys
if sys.platform == "win32":
    project_root = Path(r"F:/MScPROJECT_LOCAL")
else:
    project_root = Path("/workspaces/MScPROJECT_LOCAL")
case_name = "07_tropea_cafe"
experiment = "cat"

#These neeed work
case_dir     = project_root / "Data" / case_name
source_dir   = case_dir / "010_source"
yolo_masks_dir = case_dir / "015_YOLO" / "cat_02b"
frames_for_recon   = case_dir / "020_frames_for_recon" / "allwalk25int" 
#frames_for_recon   = case_dir / "019_CUT3R_Frames"
reg_dir      = case_dir / "030_Registrations"
csv_path_GPS = case_dir / "000_GPS_Data" / "GPS_tags.csv"
csv_path_pose = reg_dir /"camera_poses.csv"

#TEMP
#This needs to be filed after user input decides on what to analyse
ppt_template = '01_test_02'

# video loading
# Auto-find the video in 010_source
videos = list(source_dir.glob("*.mp4"))
assert len(videos) == 1, f"Expected 1 video, found {len(videos)}: {videos}"
video_path = videos[0]
print(f"Case    : {case_name}")
print(f"Video   : {video_path.name}")


Case    : 07_tropea_cafe
Video   : VID_20260530_113822990.mp4


In [ ]:
#stabilisation - basic warps
import subprocess

src = video_path
stab_dir = case_dir / "011_stabilised"
stab_dir.mkdir(exist_ok=True)
cfr  = stab_dir / f"{case_name}_cfr.mp4"
trf  = stab_dir / f"{case_name}.trf"
out  = stab_dir / f"{case_name}_stab.mp4"
ffmpeg = r"C:\ffmpeg\bin\ffmpeg.exe" if sys.platform == "win32" else "ffmpeg"

# Parameters
fps        = 30 # frame rate to encode
crf        = 18 # quality
shakiness  = 10 # size of match window
accuracy   = 15 # iterations on matcing and cam solve
smoothing  = 20 # gaussian smoothing on cam path
zoom       = 5 # crop and zoom factor

# CFR conversion
subprocess.run([ffmpeg, "-i", str(src), "-vf", f"fps={fps}", "-c:v", "libx264", "-crf", str(crf), "-c:a", "copy", str(cfr)], check=True)

# Pass 1
subprocess.run([ffmpeg, "-i", str(cfr), "-vf", f"vidstabdetect=shakiness={shakiness}:accuracy={accuracy}:result={trf}", "-f", "null", "-"], check=True)

# Pass 2
subprocess.run([ffmpeg, "-i", str(cfr), "-vf", f"vidstabtransform=input={trf}:smoothing={smoothing}:zoom={zoom}:crop=black", "-c:v", "libx264", "-crf", str(crf), "-c:a", "copy", str(out)], check=True)
video_stab = str(out)

In [ ]:
#bypass stab to run cut3r on already stabilised
stab_dir = case_dir / "011_stabilised"
out  = stab_dir / f"{case_name}_stab_01.mp4"
video_stab = str(out)

In [ ]:
#YOLO - track moving subject on the stabilised (locked-camera) footage
#works from video


import sys
sys.path.insert(0, str(project_root / "src"))

from A_YOLO_seg import track_subject_masks

results = track_subject_masks(
    video_path  = video_path,
    output_dir  = str(yolo_masks_dir),
    subjects    = ["cat"],
    start_frame = 3777,
    end_frame   = 4600,
    conf = 0.05
)

print(f"\n{len(results)} masks written to {yolo_masks_dir}")


WARNING ⚠️ imgsz=[1080, 1920] must be multiple of max stride 32, updating to [1088, 1920]
WARNING ⚠️ imgsz=[1080, 1920] must be multiple of max stride 32, updating to [1088, 1920]
WARNING ⚠️ imgsz=[1080, 1920] must be multiple of max stride 32, updating to [1088, 1920]
WARNING ⚠️ imgsz=[1080, 1920] must be multiple of max stride 32, updating to [1088, 1920]
WARNING ⚠️ imgsz=[1080, 1920] must be multiple of max stride 32, updating to [1088, 1920]
WARNING ⚠️ imgsz=[1080, 1920] must be multiple of max stride 32, updating to [1088, 1920]
WARNING ⚠️ imgsz=[1080, 1920] must be multiple of max stride 32, updating to [1088, 1920]
WARNING ⚠️ imgsz=[1080, 1920] must be multiple of max stride 32, updating to [1088, 1920]
WARNING ⚠️ imgsz=[1080, 1920] must be multiple of max stride 32, updating to [1088, 1920]
WARNING ⚠️ imgsz=[1080, 1920] must be multiple of max stride 32, updating to [1088, 1920]
WARNING ⚠️ imgsz=[1080, 1920] must be multiple of max stride 32, updating to [1088, 1920]
WARNING ⚠️

In [5]:
#FRAME SELECTION - Frame selection to make the recon
CUT3R_size = (512,288) # resolution
# Parameters
fps = 30.07
interval     = 4/fps      # seconds between samples - expressed as frames to skip, per second(frame rate)
window       = 15     # search window for sharpness
min_sharp    = 0.0    # raise this after inspecting selection_log.json
res          =   CUT3R_size # resolution

import sys
sys.path.insert(0, str(project_root / "src"))

from B_video_processing import image_sequencer

results = image_sequencer(
    #video_path    = str(video_path),
    video_path = str(video_path),
    output_dir    = str(frames_for_recon),
    #res           = res,
    interval_sec  = interval,
    search_window = window,
    min_sharpness = min_sharp,
    start_frame  = 1600,
    end_frame    = 1700
)

print(f"\n{len(results)} frames written to {frames_for_recon}")


Window  : clamped from ±15 to ±2 (half interval)
Video   : VID_20260530_113822990.mp4
          30.1 fps · 9675 frames · 321.8s
Range   : frames 1600-1700 (100 frames)
Sampling: every 4 frames (0.13302294645826404s) → 25 candidates
Window  : ±2 frames  |  scoring at 25% resolution


Pass 1: scoring: 100%|██████████| 100/100 [00:00<00:00, 156.46fr/s]


Selected: 25 frames  (13 shifted from sample point, 0 skipped below threshold)


Pass 3: writing: 100%|██████████| 100/100 [00:02<00:00, 36.08fr/s]


Log     : /workspaces/MScPROJECT_LOCAL/Data/07_tropea_cafe/020_frames_for_recon/allwalk25int_extracorner_v02/selection_log.json
Sharpness (at 25% res): min 1148 · mean 2476 · max 3872

2 frames written to /workspaces/MScPROJECT_LOCAL/Data/07_tropea_cafe/020_frames_for_recon/allwalk25int_extracorner_v02


In [ ]:
#CUT3R basic implementation - copy past into command line
'''
cd c:\Users\djbar\Documents\MScPROJECT_LOCAL\Models\CUT3R
python demo.py --model_path src/cut3r_512_dpt_4_64.pth --seq_path examples/001 --size 512 --vis_threshold 1.5 --output_dir tmp
'''

In [ ]:
# Cell — CUT3R reconstruction + 3rd-person novel view synthesis
import importlib, sys, traceback
sys.path.insert(0, str(project_root / "src"))

import B_cut3r_recon; importlib.reload(B_cut3r_recon)
from B_cut3r_recon import run_cut3r
import torch, gc
gc.collect()
torch.cuda.empty_cache()


try:
    run_cut3r(
        frames_dir       = frames_for_recon,
        output_dir       = case_dir / "035_CUT3R_output" / "01_walk_sparse_50_02_full",
        ckpt_path        = project_root / "Models" / "CUT3R" / "src" / "cut3r_512_dpt_4_64.pth",
        render_3rdperson = True,
        back_dist        = 0.5,
        up_dist          = 0.3,
        revisit          = False,
        conf_threshold   = 1.5,
    )
except Exception:
    traceback.print_exc()


In [ ]:
#camera assesment CUT3R
data = np.load(r"F:\MScPROJECT_LOCAL\Data\01_walk\035_CUT3R_output\24\camera\000100.npz")
data["pose"]        # 4×4 camera-to-world
#data["intrinsics"]  # 3×3 K matrix


In [ ]:
# Cell — CUT3R visualisation
import importlib, sys
sys.path.insert(0, str(project_root / "src"))

import B_cut3r_vis; importlib.reload(B_cut3r_vis)
from B_cut3r_vis import visualise_cut3r

visualise_cut3r(case_dir / "035_CUT3R_output" / "01_walk_sparse_50_02_full")


In [ ]:
# Cell — CUT3R post-hoc novel view synthesis (requires cut3r_state.pt)
import importlib, sys, traceback
sys.path.insert(0, str(project_root / "src"))

import B_cut3r_recon; importlib.reload(B_cut3r_recon)
from B_cut3r_recon import query_nvs

try:
    query_nvs(
        output_dir  = case_dir / "013_CUT3R_output",
        ckpt_path   = project_root / "Models" / "CUT3R" / "src" / "cut3r_512_dpt_4_64.pth",
        back_dist   = 0.5,
        up_dist     = 0.3,
        out_subdir  = "nvs_rgb",
    )
except Exception:
    traceback.print_exc()

In [ ]:
# Cell — MegaSaM reconstruction (mono-depth -> camera tracking -> CVD optimisation)
# requires the `mega_sam` conda env (Linux devcontainer only, see
# Models/mega-sam/LINUX_SETUP_CHANGELOG.md) — not available if running this
# notebook natively on Windows.
'''this will replace CUT3R probably'''
import importlib, sys, traceback
sys.path.insert(0, str(project_root / "src"))

import B_megasam_recon; importlib.reload(B_megasam_recon)
from B_megasam_recon import run_megasam

mega_sam_dir = project_root / "Models" / "mega-sam"

try:
    megasam_npz = run_megasam(
        frames_dir   = frames_for_recon,
        output_dir   = case_dir / "036_MEGASAM_output" / "01_walk_sparse_50_02_full",
        mega_sam_dir = mega_sam_dir,
        scene_name   = case_name,
    )
except Exception:
    traceback.print_exc()


In [ ]:
# Cell — MegaSaM visualisation
import importlib, sys
sys.path.insert(0, str(project_root / "src"))

import B_megasam_vis; importlib.reload(B_megasam_vis)
from B_megasam_vis import visualise_megasam

visualise_megasam(megasam_npz)


In [3]:
# Cell — VGGT reconstruction
import sys
import argparse
sys.path.insert(0, str(project_root / "Models" / "VGGT"/ "vggt"))

from demo_colmap import demo_fn

recon_out  = case_dir / "034_VGGT_output" 
max_frames = 55

args = argparse.Namespace(
    scene_dir        = str(case_dir),
   
    image_dir        = str(frames_for_recon),
    output_dir       = str(recon_out),
    max_frames       = max_frames,
    load_resolution  = 518,
    seed             = 42,
    use_ba           = False,
    max_reproj_error = 8.0,
    shared_camera    = False,
    camera_type      = "SIMPLE_PINHOLE",
    vis_thresh       = 0.2,
    query_frame_num  = 8,
    max_query_pts    = 4096,
    fine_tracking    = True,
    conf_thres_value = 5.0,
)

demo_fn(args)


/opt/conda/envs/Msc2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Arguments: {'scene_dir': '/workspaces/MScPROJECT_LOCAL/Data/01_walk', 'image_dir': '/workspaces/MScPROJECT_LOCAL/Data/01_walk/020_frames_for_recon/01_walk_sparse_50_02_full', 'output_dir': '/workspaces/MScPROJECT_LOCAL/Data/01_walk/034_VGGT_output', 'max_frames': 55, 'load_resolution': 518, 'seed': 42, 'use_ba': False, 'max_reproj_error': 8.0, 'shared_camera': False, 'camera_type': 'SIMPLE_PINHOLE', 'vis_thresh': 0.2, 'query_frame_num': 8, 'max_query_pts': 4096, 'fine_tracking': True, 'conf_thres_value': 5.0}
Setting seed as: 42
Using device: cuda
Using dtype: torch.bfloat16
Model loaded
Loaded 50 images from /workspaces/MScPROJECT_LOCAL/Data/01_walk/020_frames_for_recon/01_walk_sparse_50_02_full
Running aggregator on 50 frames at 518px...


/workspaces/MScPROJECT_LOCAL/Models/VGGT/vggt/demo_colmap.py:80: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=dtype):


RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [17]:
#VGGT output conversion
from P_projection_mapping import build_frame_index
FRAME_NAME_FMT = "{:04d}.jpg"
frame_to_row = build_frame_index(frames_for_recon, name_fmt=FRAME_NAME_FMT)

In [1]:
#convert model-world-space into real-world-space
#get lat lon positions
#get transforms to convert other things too
from Q_Metric_georeferencing import model_to_GPS_calibrated_locations
All_poses_realworld, geo_cords, R, s, t = model_to_GPS_calibrated_locations (csv_path_GPS, csv_path_pose)

NameError: name 'csv_path_GPS' is not defined

In [ ]:
import sys
print(sys.executable)


In [ ]:
#Fit camera poses to realworld coordinates



In [23]:
#Projection Mapping and subject positioning
from P_projection_mapping import projection_mapping_sequence
subject_positions = projection_mapping_sequence(
    recon_out  = case_dir / "034_VGGT_output" ,
    predictions_path = case_dir /"035_VGGT_O_output" /"cat_02" /"predictions.npz",
    frames_dir = str(frames_for_recon),
    masks_dir = str(yolo_masks_dir),
    proj_out_dir = case_dir / "090_assets",
    main_idx = 3927, #view to use
    extra_indices = [ 3877 , 3890, 3927 , 3952 , 3977, 4002, 4027,  4052, 4065 ],#views to project
    conf_threshold = 1.5,
    frame_to_row = frame_to_row
    )

#subject positions are ready to be turned into map!

attempting 3877
attempting to write 3877
attempting 3890
attempting to write 3890
attempting 3927
attempting to write 3927
attempting 3952
attempting to write 3952
attempting 3977
attempting to write 3977
attempting 4002
attempting to write 4002
attempting 4027
attempting to write 4027
attempting 4052
attempting to write 4052
attempting 4065
attempting to write 4065


In [12]:
#temp bipass
predictions_path = case_dir /"035_VGGT_O_output" /"cat_02" /"predictions.npz"
frames_dir = str(frames_for_recon)

In [ ]:
#3d analysis
from subject_analysis import subject_world_positions
frame_indices = [ 3877 , 3890, 3927 , 3952 , 3977, 4002, 4027,  4052, 4065 ]
pos = subject_world_positions(frame_indices, predictions_path, frame_to_row)

In [14]:
print(pos)

{3877: tensor([0.5006, 0.0781, 0.8806], device='cuda:0'), 3890: tensor([0.4955, 0.0779, 0.8701], device='cuda:0'), 3927: tensor([0.4879, 0.0780, 0.8470], device='cuda:0'), 3952: None, 3977: tensor([0.4593, 0.0736, 0.7669], device='cuda:0'), 4002: tensor([0.4592, 0.0747, 0.7564], device='cuda:0'), 4027: tensor([0.4479, 0.0730, 0.7370], device='cuda:0'), 4052: tensor([0.4551, 0.0770, 0.7430], device='cuda:0'), 4065: tensor([0.4509, 0.0768, 0.7386], device='cuda:0')}


In [ ]:
#MAKING A MOVIE
'''This controls M_scoring. It's independent of scene recon, so could be placed anywhere'''
'''bring in M_scoring_controls'''


In [ ]:
# Cell — Populate PowerPoint from CSV
import sys
import importlib
sys.path.insert(0, str(project_root / "src"))

import W_populate_pptx
importlib.reload(W_populate_pptx)
from W_populate_pptx import populate



report_template = project_root / "src" / "Powerpoint" / f"{ppt_template}.pptx"
ppt_assets   = case_dir / "090_Assets"
ppt_csv      = ppt_assets / f"{case_name}.csv"
ppt_output   = case_dir / "100_Powerpoint" / f"{case_name}.pptx"


populate(str(report_template), str(ppt_csv), str(ppt_output), str(ppt_assets))


In [ ]:
import pycolmap, sys
print(pycolmap.__version__)
print(sys.executable)
